In [1]:
from pathlib import Path
import pandas as pd
import socket

print("Running on:", socket.gethostname())

YEAR = 2019
RAW = Path("/scratch/sm3664/medicare-fraud/raw/part_b")
name = f"Medicare_Physician_Other_Practitioners_by_Provider_and_Service_{YEAR}"
f = RAW / name / f"{name}.csv"
print(f, f.exists())

Running on: hal0345.amarel.rutgers.edu
/scratch/sm3664/medicare-fraud/raw/part_b/Medicare_Physician_Other_Practitioners_by_Provider_and_Service_2019/Medicare_Physician_Other_Practitioners_by_Provider_and_Service_2019.csv True


In [2]:
import os
print("CPUs available:", len(os.sched_getaffinity(0)))
print("Memory limit:", open("/sys/fs/cgroup/memory.max").read().strip() if os.path.exists("/sys/fs/cgroup/memory.max") else "unknown")

CPUs available: 64
Memory limit: unknown


In [3]:
CAT_COLS = ["Rndrng_NPI", "Rndrng_Prvdr_Ent_Cd", "Rndrng_Prvdr_Type", "Place_Of_Srvc", "HCPCS_Cd"]
NUM_COLS = ["Tot_Benes", "Tot_Srvcs", "Tot_Bene_Day_Srvcs", "Avg_Sbmtd_Chrg", "Avg_Mdcr_Pymt_Amt"]

df = pd.read_csv(f, usecols=CAT_COLS + NUM_COLS, dtype=str)
print(YEAR, df.shape)

2019 (10140228, 10)


In [4]:
df.isna().sum()

Rndrng_NPI             0
Rndrng_Prvdr_Ent_Cd    0
Rndrng_Prvdr_Type      0
HCPCS_Cd               0
Place_Of_Srvc          0
Tot_Benes              0
Tot_Srvcs              0
Tot_Bene_Day_Srvcs     0
Avg_Sbmtd_Chrg         0
Avg_Mdcr_Pymt_Amt      0
dtype: int64

In [5]:
for c in NUM_COLS:
    s = df[c].str.replace(r"[$,]", "", regex=True)
    bad = pd.to_numeric(s, errors="coerce").isna() & df[c].notna()
    sym = df[c].str.contains(r"[$,]", regex=True, na=False).sum()
    print(f"{c}: unparseable={bad.sum()}, has $ or ,={sym}, e.g. {df.loc[bad, c].head(3).tolist()}")

Tot_Benes: unparseable=0, has $ or ,=0, e.g. []
Tot_Srvcs: unparseable=0, has $ or ,=0, e.g. []
Tot_Bene_Day_Srvcs: unparseable=0, has $ or ,=0, e.g. []
Avg_Sbmtd_Chrg: unparseable=0, has $ or ,=0, e.g. []
Avg_Mdcr_Pymt_Amt: unparseable=0, has $ or ,=0, e.g. []


In [6]:
print("Ent_Cd:", df["Rndrng_Prvdr_Ent_Cd"].value_counts(dropna=False).to_dict())
print("Place_Of_Srvc:", df["Place_Of_Srvc"].value_counts(dropna=False).to_dict())
print("Provider types:", df["Rndrng_Prvdr_Type"].nunique())
print("HCPCS codes:", df["HCPCS_Cd"].nunique())
print("NPIs:", df["Rndrng_NPI"].nunique())

Ent_Cd: {'I': 9704223, 'O': 436005}
Place_Of_Srvc: {'O': 6252677, 'F': 3887551}
Provider types: 99
HCPCS codes: 6138
NPIs: 1093367


In [7]:
multi = df.groupby("Rndrng_NPI")["Rndrng_Prvdr_Type"].nunique()
print("NPIs with >1 provider type:", (multi > 1).sum())
print("NPIs not 10 digits:", (~df["Rndrng_NPI"].str.fullmatch(r"\d{10}", na=False)).sum())
print("Dup NPI+HCPCS+POS:", df.duplicated(["Rndrng_NPI", "HCPCS_Cd", "Place_Of_Srvc"]).sum())
print("Exact duplicates:", df.duplicated().sum())

NPIs with >1 provider type: 0
NPIs not 10 digits: 0
Dup NPI+HCPCS+POS: 0
Exact duplicates: 0


In [8]:
pd.set_option("display.max_rows", 200)
df["Rndrng_Prvdr_Type"].value_counts().sort_index()

Rndrng_Prvdr_Type
Addiction Medicine                                             903
Advanced Heart Failure and Transplant Cardiology              7966
All Other Suppliers                                            293
Allergy/ Immunology                                          29691
Ambulance Service Provider                                   32795
Ambulatory Surgical Center                                   60256
Anesthesiology                                              247306
Anesthesiology Assistant                                      5257
Audiologist                                                  21468
Cardiac Surgery                                              11467
Cardiology                                                  427443
Centralized Flu                                              52539
Certified Clinical Nurse Specialist                           7188
Certified Nurse Midwife                                       1237
Certified Registered Nurse Anesthetist (CRNA

In [9]:
import gc
del df; gc.collect()

def profile(year):
    name = f"Medicare_Physician_Other_Practitioners_by_Provider_and_Service_{year}"
    d = pd.read_csv(RAW / name / f"{name}.csv", usecols=CAT_COLS + NUM_COLS, dtype=str)
    unparse = sum(int((pd.to_numeric(d[c].str.replace(r"[$,]", "", regex=True), errors="coerce").isna() & d[c].notna()).sum()) for c in NUM_COLS)
    out = {
        "year": year,
        "rows": len(d),
        "npis": d["Rndrng_NPI"].nunique(),
        "null_cols": {c: int(n) for c, n in d.isna().sum().items() if n},
        "unparseable": unparse,
        "npi_multi_type": int((d.groupby("Rndrng_NPI")["Rndrng_Prvdr_Type"].nunique() > 1).sum()),
        "npi_not_10": int((~d["Rndrng_NPI"].str.fullmatch(r"\d{10}", na=False)).sum()),
        "dup_key": int(d.duplicated(["Rndrng_NPI", "HCPCS_Cd", "Place_Of_Srvc"]).sum()),
        "ent_O_pct": round((d["Rndrng_Prvdr_Ent_Cd"] == "O").mean() * 100, 2),
        "n_types": d["Rndrng_Prvdr_Type"].nunique(),
    }
    types = set(d["Rndrng_Prvdr_Type"].unique())
    del d; gc.collect()
    return out, types

summaries, type_sets = [], {}
for y in [2019, 2020, 2021, 2022, 2023]:
    s, t = profile(y)
    summaries.append(s)
    type_sets[y] = t
    print(s)

pd.DataFrame(summaries)

{'year': 2019, 'rows': 10140228, 'npis': 1093367, 'null_cols': {}, 'unparseable': 0, 'npi_multi_type': 0, 'npi_not_10': 0, 'dup_key': 0, 'ent_O_pct': np.float64(4.3), 'n_types': 99}
{'year': 2020, 'rows': 9449361, 'npis': 1085313, 'null_cols': {}, 'unparseable': 0, 'npi_multi_type': 0, 'npi_not_10': 0, 'dup_key': 0, 'ent_O_pct': np.float64(4.58), 'n_types': 102}
{'year': 2021, 'rows': 9886177, 'npis': 1123589, 'null_cols': {}, 'unparseable': 0, 'npi_multi_type': 0, 'npi_not_10': 0, 'dup_key': 0, 'ent_O_pct': np.float64(6.05), 'n_types': 103}
{'year': 2022, 'rows': 9755427, 'npis': 1148873, 'null_cols': {}, 'unparseable': 0, 'npi_multi_type': 0, 'npi_not_10': 0, 'dup_key': 0, 'ent_O_pct': np.float64(5.59), 'n_types': 103}
{'year': 2023, 'rows': 9660647, 'npis': 1175281, 'null_cols': {}, 'unparseable': 0, 'npi_multi_type': 0, 'npi_not_10': 0, 'dup_key': 0, 'ent_O_pct': np.float64(5.56), 'n_types': 104}


,year,rows,npis,null_cols,unparseable,npi_multi_type,npi_not_10,dup_key,ent_O_pct,n_types
0,2019,10140228,1093367,{},0,0,0,0,4.30,99
1,2020,9449361,1085313,{},0,0,0,0,4.58,102
2,2021,9886177,1123589,{},0,0,0,0,6.05,103
3,2022,9755427,1148873,{},0,0,0,0,5.59,103
4,2023,9660647,1175281,{},0,0,0,0,5.56,104


In [10]:
all_types = sorted(set().union(*type_sets.values()))
print("Union of provider types across 2019-2023:", len(all_types))

presence = pd.DataFrame({y: [t in type_sets[y] for t in all_types] for y in type_sets}, index=all_types)
partial = presence[~presence.all(axis=1)]
print("Types missing from at least one year:", len(partial))
partial

Union of provider types across 2019-2023: 104
Types missing from at least one year: 5


,2019,2020,2021,2022,2023
Adult Congenital Heart Disease,False,True,True,True,True
Dental Anesthesiology,False,False,False,False,True
Home Infusion Therapy Services,False,False,True,True,True
Micrographic Dermatologic Surgery,False,True,True,True,True
Opioid Treatment Program,False,True,True,True,True


In [11]:
import re
norm = lambda s: re.sub(r"[^a-z]", "", s.lower())
groups = {}
for t in all_types:
    groups.setdefault(norm(t), []).append(t)
print({k: v for k, v in groups.items() if len(v) > 1})

{}
